# 03 Cechy – agregacja tygodniowa i fazowa

## 1. Konfiguracja

In [1]:
import sys
sys.path.append("..")

import pandas as pd

from src.config import EMOTIONS, PROCESSED_DIR

NEGATIVE = ["anger", "disgust", "fear", "sadness"]
keys = ["subreddit", "camp", "cycle", "week_start", "phase"]

## 2. Połączenie próby z wynikami modeli

In [2]:
sample = pd.read_parquet(PROCESSED_DIR / "sample.parquet", columns=["id", "subreddit", "camp", "cycle", "week_start", "phase", "n_words"])
emotions = pd.concat(pd.read_parquet(path) for path in sorted((PROCESSED_DIR / "emotions").glob("*.parquet")))
sentiment = pd.concat(pd.read_parquet(path) for path in sorted((PROCESSED_DIR / "sentiment").glob("*.parquet")))
data = sample.merge(emotions, on="id").merge(sentiment.add_prefix("sent_").rename(columns={"sent_id": "id"}), on="id")
data["sent_neg"] = data[["sent_negative", "sent_neutral", "sent_positive"]].idxmax(axis=1).eq("sent_negative")
data["sent_pos"] = data[["sent_negative", "sent_neutral", "sent_positive"]].idxmax(axis=1).eq("sent_positive")
print(len(sample), len(data))

794519 794519


## 3. Agregacja tygodniowa i fazowa (próba główna i próg 10 słów)

In [3]:
measures = EMOTIONS + ["negative", "positive", "arousal", "sent_neg_share", "sent_pos_share"]
for suffix, subset in {"": data, "_10w": data[data["n_words"] >= 10]}.items():
    grouped = subset.groupby(keys, dropna=False)
    weekly = grouped[EMOTIONS].mean()
    weekly.insert(0, "n", grouped.size())
    weekly["negative"] = weekly[NEGATIVE].sum(axis=1)
    weekly["positive"] = weekly["joy"]
    weekly["arousal"] = (weekly["anger"] + weekly["fear"]) / (weekly["anger"] + weekly["fear"] + weekly["sadness"])
    weekly["sent_neg_share"] = grouped["sent_neg"].mean()
    weekly["sent_pos_share"] = grouped["sent_pos"].mean()
    weekly = weekly.reset_index()
    weekly.to_csv(PROCESSED_DIR / f"weekly_emotions{suffix}.csv", index=False)

    by_phase = weekly.dropna(subset=["phase"]).groupby(["subreddit", "camp", "cycle", "phase"])
    phases = by_phase[measures].mean()
    phases.insert(0, "n", by_phase["n"].sum())
    phases.insert(0, "n_weeks", by_phase.size())
    phases.reset_index().to_csv(PROCESSED_DIR / f"phase_emotions{suffix}.csv", index=False)

## 4. Podgląd

In [4]:
weekly = pd.read_csv(PROCESSED_DIR / "weekly_emotions.csv", parse_dates=["week_start"])
weekly.head()

,subreddit,camp,cycle,week_start,phase,n,anger,disgust,fear,joy,neutral,sadness,surprise,negative,positive,arousal,sent_neg_share,sent_pos_share
0,Conservative,rep,2012,2011-03-21,bazowa,255,0.142672,0.182389,0.030192,0.041698,0.467701,0.027403,0.107947,0.382655,0.041698,0.863168,0.611765,0.082353
1,Conservative,rep,2012,2011-03-28,bazowa,300,0.181040,0.198051,0.019124,0.035462,0.447910,0.032911,0.085503,0.431126,0.035462,0.858797,0.646667,0.056667
2,Conservative,rep,2012,2011-04-04,bazowa,273,0.153721,0.172175,0.024587,0.044202,0.462873,0.041889,0.100554,0.392372,0.044202,0.809766,0.641026,0.069597
3,Conservative,rep,2012,2011-04-11,bazowa,300,0.151041,0.213465,0.032411,0.039358,0.451064,0.036914,0.075746,0.433832,0.039358,0.832487,0.630000,0.056667
4,Conservative,rep,2012,2011-04-18,bazowa,259,0.141620,0.206523,0.039285,0.051229,0.444154,0.039849,0.077340,0.427277,0.051229,0.819486,0.567568,0.084942


In [5]:
pd.read_csv(PROCESSED_DIR / "phase_emotions.csv")[["subreddit", "cycle", "phase", "n_weeks", "n", "negative", "arousal"]].head(10)

,subreddit,cycle,phase,n_weeks,n,negative,arousal
0,Conservative,2012,bazowa,12,3400,0.402216,0.820947
1,Conservative,2012,powyborcza,8,2400,0.384043,0.797633
2,Conservative,2012,przedwyborcza,64,19200,0.393651,0.816995
3,Conservative,2012,przejsciowa,11,3300,0.369428,0.791305
4,Conservative,2012,szczytowa,9,2700,0.364150,0.775265
5,Conservative,2016,bazowa,19,5700,0.413940,0.808915
6,Conservative,2016,powyborcza,8,2400,0.404652,0.830492
7,Conservative,2016,przedwyborcza,50,15000,0.400037,0.806926
8,Conservative,2016,przejsciowa,11,3300,0.407410,0.841315
9,Conservative,2016,szczytowa,16,4800,0.415540,0.815829
